In [1]:
!pip install torch torchvision

   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   --------------- ------------------------ 0.5/1.4 MB 2.4 MB/s eta 0:00:01
   ------------------------------ --------- 1.0/1.4 MB 2.5 MB/s eta 0:00:01
   ---------------------------------------- 1.4/1.4 MB 2.5 MB/s  0:00:00


In [3]:
!pip install ultralytics

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.5 MB ? eta -:--:--
   --------------------- ------------------ 0.8/1.5 MB 1.6 MB/s eta 0:00:01
   ------------------------------------ --- 1.3/1.5 MB 2.0 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 1.7 MB/s  0:00:00
   ---------------------------------------- 0.0/865.8 kB ? eta -:--:--
   ------------------------ --------------- 524.3/865.8 kB 4.2 MB/s eta 0:00:01
   ---------------------------------------- 865.8/865.8 kB 3.9 MB/s  0:00:00
   ---------------------------------------- 0.0/51.3 MB ? eta -:--:--
   ---------------------------------------- 0.3/51.3 MB ? eta -:--:--
    --------------------------------------- 1.0/51.3 MB 3.4 MB/s eta 0:00:15
   - -------------------------------------- 2.1/51.3 MB 3.9 MB/s eta 0:00:13
   -- ------------------------------------- 2.

In [12]:
#!/usr/bin/env python3
"""
Single-File High-Throughput Vision Pipeline
Technologies: PyTorch, ONNX, NVIDIA TensorRT 10.x, CUDA Streams, Torchvision GPU NMS
"""

import os
import sys
import time
from typing import Dict, List, Tuple

import numpy as np
import torch
import torchvision
from ultralytics import YOLO
import tensorrt as trt

# Initialize TRT Logger
TRT_LOGGER = trt.Logger(trt.Logger.WARNING)


# =====================================================================
# 1. MODEL EXPORT (PyTorch -> ONNX)
# =====================================================================
def export_onnx_model(weights: str = "yolov8s.pt", onnx_path: str = "yolov8s.onnx") -> str:
    """Exports a standard YOLO model to dynamic-batch ONNX format."""
    if os.path.exists(onnx_path):
        print(f"[+] ONNX model already exists at: {onnx_path}")
        return onnx_path

    print(f"[+] Exporting {weights} to ONNX with dynamic batch support...")
    model = YOLO(weights)
    exported_file = model.export(
        format="onnx",
        dynamic=True,
        opset=17,
        simplify=True
    )
    if os.path.abspath(exported_file) != os.path.abspath(onnx_path):
        os.rename(exported_file, onnx_path)
    print(f"[✓] Successfully exported to {onnx_path}")
    return onnx_path
    
# =====================================================================
# 2. TENSORRT ENGINE BUILDER
# =====================================================================
def build_trt_engine(
    onnx_path: str,
    engine_path: str,
    fp16: bool = True,
    max_batch: int = 16,
    opt_batch: int = 4,
    workspace_gb: int = 4
) -> str:
    """Parses ONNX and builds an optimized serialized TensorRT engine."""
    if os.path.exists(engine_path):
        print(f"[+] TensorRT engine already exists at: {engine_path}")
        return engine_path

    print(f"[+] Compiling TensorRT Engine from {onnx_path}...")
    builder = trt.Builder(TRT_LOGGER)
    config = builder.create_builder_config()

    flag = 1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH)
    network = builder.create_network(flag)
    parser = trt.OnnxParser(network, TRT_LOGGER)

    with open(onnx_path, "rb") as model_file:
        if not parser.parse(model_file.read()):
            for err in range(parser.num_errors):
                print(f"[!] Parser Error: {parser.get_error(err)}")
            raise RuntimeError("Failed to parse ONNX model.")

    # Dynamic shape profile (Batch, Channels, Height, Width)
    profile = builder.create_optimization_profile()
    input_tensor = network.get_input(0)
    input_name = input_tensor.name

    profile.set_shape(
        input_name,
        min=(1, 3, 640, 640),
        opt=(opt_batch, 3, 640, 640),
        max=(max_batch, 3, 640, 640)
    )
    config.add_optimization_profile(profile)

    # Set GPU memory pool limit
    config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, workspace_gb * (1024 ** 3))

    if fp16 and builder.platform_has_fast_fp16:
        print("[+] Enabling Fast FP16 Mode")
        config.set_flag(trt.BuilderFlag.FP16)

    print("[+] Building serialized engine (this may take a few minutes)...")
    serialized_engine = builder.build_serialized_network(network, config)
    if serialized_engine is None:
        raise RuntimeError("Failed to build TensorRT engine.")

    with open(engine_path, "wb") as f:
        f.write(serialized_engine)
    print(f"[✓] Engine built and saved to: {engine_path}")
    return engine_path


# =====================================================================
# 3. HIGH-PERFORMANCE ZERO-COPY INFERENCE RUNNER
# =====================================================================
class TensorRTInferenceEngine:
    """Manages TensorRT 10.x runtime execution directly with Torch CUDA tensors."""
    def __init__(self, engine_path: str, device: str = "cuda:0"):
        self.device = torch.device(device)
        self.runtime = trt.Runtime(TRT_LOGGER)
        
        with open(engine_path, "rb") as f:
            self.engine = self.runtime.deserialize_cuda_engine(f.read())

        self.context = self.engine.create_execution_context()
        self.stream = torch.cuda.Stream(device=self.device)

        self.input_names: List[str] = []
        self.output_names: List[str] = []
        self.tensor_dtypes: Dict[str, torch.dtype] = {}

        trt_to_torch = {
            trt.DataType.FLOAT: torch.float32,
            trt.DataType.HALF: torch.float16,
            trt.DataType.INT32: torch.int32,
            trt.DataType.INT8: torch.int8,
            trt.DataType.BOOL: torch.bool
        }

        for i in range(self.engine.num_io_tensors):
            name = self.engine.get_tensor_name(i)
            mode = self.engine.get_tensor_mode(name)
            dtype = self.engine.get_tensor_dtype(name)
            self.tensor_dtypes[name] = trt_to_torch.get(dtype, torch.float32)

            if mode == trt.TensorIOMode.INPUT:
                self.input_names.append(name)
            else:
                self.output_names.append(name)

    def __call__(self, input_tensor: torch.Tensor) -> Dict[str, torch.Tensor]:
        assert input_tensor.is_cuda, "Input tensor must reside on CUDA device."
        input_name = self.input_names[0]

        # Configure dynamic input shape and register memory pointer
        self.context.set_input_shape(input_name, tuple(input_tensor.shape))
        self.context.set_tensor_address(input_name, input_tensor.data_ptr())

        outputs = {}
        for out_name in self.output_names:
            out_shape = tuple(self.context.get_tensor_shape(out_name))
            out_tensor = torch.empty(out_shape, dtype=self.tensor_dtypes[out_name], device=self.device)
            self.context.set_tensor_address(out_name, out_tensor.data_ptr())
            outputs[out_name] = out_tensor

        # Async execution on dedicated CUDA stream
        with torch.cuda.stream(self.stream):
            self.context.execute_async_v3(stream_handle=self.stream.cuda_stream)
        self.stream.synchronize()

        return outputs


# =====================================================================
# 4. GPU-ACCELERATED NON-MAXIMUM SUPPRESSION (NMS)
# =====================================================================
def postprocess_gpu(
    predictions: torch.Tensor,
    conf_thresh: float = 0.25,
    iou_thresh: float = 0.45
) -> List[Tuple[torch.Tensor, torch.Tensor, torch.Tensor]]:
    """Performs batched NMS entirely in CUDA VRAM without CPU-GPU thrashing."""
    # Transpose shape: (Batch, 84, 8400) -> (Batch, 8400, 84)
    preds = predictions.transpose(1, 2)
    batch_results = []

    for pred in preds:
        boxes_cxcywh = pred[:, :4]
        class_scores = pred[:, 4:]

        scores, class_ids = torch.max(class_scores, dim=1)
        valid_mask = scores > conf_thresh

        valid_boxes = boxes_cxcywh[valid_mask]
        valid_scores = scores[valid_mask]
        valid_classes = class_ids[valid_mask]

        if valid_boxes.numel() == 0:
            batch_results.append((
                torch.empty((0, 4), device=pred.device),
                torch.empty(0, device=pred.device),
                torch.empty(0, device=pred.device)
            ))
            continue

        # Convert [cx, cy, w, h] to [x1, y1, x2, y2]
        x1 = valid_boxes[:, 0] - valid_boxes[:, 2] / 2
        y1 = valid_boxes[:, 1] - valid_boxes[:, 3] / 2
        x2 = valid_boxes[:, 0] + valid_boxes[:, 2] / 2
        y2 = valid_boxes[:, 1] + valid_boxes[:, 3] / 2
        boxes_xyxy = torch.stack([x1, y1, x2, y2], dim=-1)

        # Vectorized GPU Batched NMS
        keep_indices = torchvision.ops.batched_nms(
            boxes_xyxy, valid_scores, valid_classes, iou_thresh
        )

        batch_results.append((
            boxes_xyxy[keep_indices],
            valid_scores[keep_indices],
            valid_classes[keep_indices]
        ))

    return batch_results


# =====================================================================
# 5. BENCHMARKING & VALIDATION PIPELINE
# =====================================================================
def run_benchmark(
    weights_path: str = "yolov8s.pt",
    onnx_path: str = "yolov8s.onnx",
    engine_path: str = "yolov8s_fp16.engine",
    batch_size: int = 4,
    warmup: int = 50,
    runs: int = 200
):
    if not torch.cuda.is_available():
        print("[!] Error: CUDA device not detected. An NVIDIA GPU is required.")
        sys.exit(1)

    device_name = torch.cuda.get_device_name(0)
    print(f"[+] Initializing benchmark on GPU: {device_name}")

    # Build artifacts if not already present
    export_onnx_model(weights_path, onnx_path)
    build_trt_engine(onnx_path, engine_path, fp16=True, opt_batch=batch_size)

    # Standard synthetic input tensor
    dummy_input = torch.randn(batch_size, 3, 640, 640, dtype=torch.float32, device="cuda")

    # Benchmarking PyTorch Baseline
    print("\n[+] Benchmarking PyTorch Eager (FP32)...")
    pt_model = YOLO(weights_path).model.to("cuda").eval()
    with torch.no_grad():
        for _ in range(warmup):
            _ = pt_model(dummy_input)
        torch.cuda.synchronize()

        pt_latencies = []
        torch.cuda.reset_peak_memory_stats()
        for _ in range(runs):
            t0 = time.perf_counter()
            _ = pt_model(dummy_input)
            torch.cuda.synchronize()
            pt_latencies.append((time.perf_counter() - t0) * 1000)
        pt_peak_vram = torch.cuda.max_memory_allocated() / (1024 ** 2)

    # Benchmarking TensorRT 10.x Engine + GPU NMS
    print("[+] Benchmarking TensorRT 10.x (FP16) + GPU NMS...")
    trt_runner = TensorRTInferenceEngine(engine_path)
    for _ in range(warmup):
        _ = trt_runner(dummy_input)
    torch.cuda.synchronize()

    trt_latencies = []
    torch.cuda.reset_peak_memory_stats()
    for _ in range(runs):
        t0 = time.perf_counter()
        raw_outputs = trt_runner(dummy_input)
        first_output_key = list(raw_outputs.keys())[0]
        _ = postprocess_gpu(raw_outputs[first_output_key])
        torch.cuda.synchronize()
        trt_latencies.append((time.perf_counter() - t0) * 1000)
    trt_peak_vram = torch.cuda.max_memory_allocated() / (1024 ** 2)

    # Output formatted report
    pt_med = np.median(pt_latencies)
    pt_p95 = np.percentile(pt_latencies, 95)
    pt_fps = (batch_size / (pt_med / 1000))

    trt_med = np.median(trt_latencies)
    trt_p95 = np.percentile(trt_latencies, 95)
    trt_fps = (batch_size / (trt_med / 1000))

    print("\n" + "=" * 80)
    print(f"{'BENCHMARK RESULTS':^80}")
    print(f"{'Target GPU: ' + device_name:^80}")
    print("=" * 80)
    print(f"{'Metric':<30} | {'PyTorch (FP32)':<20} | {'TensorRT 10.x (FP16) + NMS':<20}")
    print("-" * 80)
    print(f"{'Median Latency':<30} | {pt_med:<17.2f} ms | {trt_med:<17.2f} ms")
    print(f"{'p95 Latency':<30} | {pt_p95:<17.2f} ms | {trt_p95:<17.2f} ms")
    print(f"{'Throughput':<30} | {pt_fps:<16.1f} FPS | {trt_fps:<16.1f} FPS")
    print(f"{'Peak VRAM':<30} | {pt_peak_vram:<17.1f} MB | {trt_peak_vram:<17.1f} MB")
    print("=" * 80)
    print(f"[✓] Speedup: {pt_med / trt_med:.2f}x faster inference with zero-copy execution.")


if __name__ == "__main__":
    run_benchmark()

[!] Error: CUDA device not detected. An NVIDIA GPU is required.


SystemExit: 1

In [13]:
import os

os.makedirs("customer-support-ai", exist_ok=True)

print("Project folder created!")

Project folder created!


In [14]:
import pandas as pd

data = {
    "ticket": [
        "My payment was deducted but my order was cancelled",
        "I cannot login to my account",
        "My package has not arrived yet",
        "I want to request a refund",
        "The application keeps crashing",
        "How can I change my password?",
        "My delivery is delayed",
        "I was charged twice for the same order",
        "The website is very slow",
        "I want to cancel my subscription",
        "I am very happy with your service",
        "Please help me update my email address",
        "My refund has not arrived",
        "My payment failed",
        "How can I track my order?",
        "The app is showing an error",
        "I want to upgrade my subscription",
        "Your service is excellent",
        "My order arrived damaged",
        "I need a refund for my purchase"
    ],

    "category": [
        "Payment",
        "Account",
        "Delivery",
        "Refund",
        "Technical",
        "Account",
        "Delivery",
        "Payment",
        "Technical",
        "Subscription",
        "Feedback",
        "Account",
        "Refund",
        "Payment",
        "Delivery",
        "Technical",
        "Subscription",
        "Feedback",
        "Delivery",
        "Refund"
    ],

    "priority": [
        "High",
        "High",
        "Medium",
        "High",
        "High",
        "Low",
        "Medium",
        "High",
        "Medium",
        "Medium",
        "Low",
        "Low",
        "High",
        "High",
        "Low",
        "High",
        "Low",
        "Low",
        "High",
        "High"
    ]
}

df = pd.DataFrame(data)

df.to_csv(
    "customer-support-ai/customer_tickets.csv",
    index=False
)

print("CSV file created successfully!")
print(df.head())

CSV file created successfully!
                                              ticket   category priority
0  My payment was deducted but my order was cance...    Payment     High
1                       I cannot login to my account    Account     High
2                     My package has not arrived yet   Delivery   Medium
3                         I want to request a refund     Refund     High
4                     The application keeps crashing  Technical     High


In [15]:
import os

print(os.listdir("customer-support-ai"))

['customer_tickets.csv']
